In [ ]:
# MiniTienda - Registro y análisis de ventas

**Materia:** Arquitectura de Computadoras y Sistemas Operativos
**Alumno:** Mike Quijije Chele
**Fecha:** 18/08/2026
**Profesor:** Wernher Braun Tellez Goomez

Sistema de consola para gestión de ventas con análisis de datos. Incluye catálogo con
tuplas, precios y stock con diccionarios, registro de ventas con listas, guardado/carga
en CSV, análisis con Pandas y NumPy, gráficos con Matplotlib y un menú interactivo
completo. Incluye los retos A (agregar producto) y B (exportar gráfico a PNG).

## 0. Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import os

%matplotlib inline

## 1. Configuración inicial (tuplas, diccionarios, listas)

`CATALOGO` es una tupla con los nombres de los productos; la posición de cada nombre
funciona como su ID. `PRECIOS` y `STOCK` son diccionarios indexados por ese mismo ID,
porque cambian con cada venta. `ventas_buffer` es la lista donde se acumulan las ventas
registradas durante la ejecución.

In [ ]:
CATALOGO = (
    "Laptop Dell XPS", # ID: 0
    "Mouse Logitech", # ID: 1
    "Teclado Mecánico", # ID: 2
    "Monitor Samsung", # ID: 3
    "Audífonos Sony" # ID: 4
)


PRECIOS = {
    0: 1200.00,
    1: 45.50,
    2: 89.90,
    3: 350.00,
    4: 120.00
}

STOCK = {
    0: 10,
    1: 50,
    2: 30,
    3: 15,
    4: 25
}


ventas_buffer = []

## 2. Funciones del sistema

### 2.1 Mostrar catálogo

In [ ]:
def mostrar_catalogo():
    """Muestra el catálogo completo con precios y stock"""
    print("\n" + "="*60)
    print(f"{'ID':^5} {'PRODUCTO':^30} {'PRECIO':^12} {'STOCK':^8}")
    print("="*60)
    for i, producto in enumerate(CATALOGO):
        if i in PRECIOS and i in STOCK:
            print(f"{i:^5} {producto:<30} \${PRECIOS[i]:>8.2f} {STOCK[i]:>8}")
    print("="*60)

<>:8: SyntaxWarning: invalid escape sequence '\$'
<>:8: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_563/2430127700.py:8: SyntaxWarning: invalid escape sequence '\$'
  print(f"{i:^5} {producto:<30} \${PRECIOS[i]:>8.2f} {STOCK[i]:>8}")


### 2.2 Registrar venta

Usa `try/except/else/finally` completo: el `try` valida y calcula, los `except` atrapan
errores de validación (`ValueError`) o cualquier otro error inesperado, el `else` solo
corre si no hubo ningún error (ahí recién se descuenta el stock y se guarda la venta), y
el `finally` siempre se ejecuta al final, haya salido bien o mal la operación.

In [ ]:
def registrar_venta():
    """Registra una venta con validaciones"""
    producto_id = None
    try:
        mostrar_catalogo()
        
        producto_id = int(input("\nID del producto: "))
        
        if producto_id not in PRECIOS:
            raise ValueError(f"Producto ID {producto_id} no existe en el catálogo")
        
        cantidad = int(input("Cantidad: "))
        
        if cantidad <= 0:
            raise ValueError("La cantidad debe ser positiva")
        
        # Verificar stock
        if cantidad > STOCK[producto_id]:
            raise ValueError(f"Stock insuficiente. Disponible: {STOCK[producto_id]}")
        
        precio_unitario = PRECIOS[producto_id]
        subtotal = precio_unitario * cantidad
        descuento = 0
        
        if cantidad >= 10:
            descuento = subtotal * 0.05 
            print(f"Descuento del 5% aplicado: \${descuento:.2f}")
        
        total = subtotal - descuento
        
    except ValueError as e:
        print(f"\n Error: {e}")
        # Registrar intento fallido en log (RETO D)
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR: {e}\n")
    except Exception as e:
        print(f"\n Error inesperado: {e}")
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR INESPERADO: {e}\n")
    else:
        # Este bloque solo corre si no hubo ningun error arriba
        STOCK[producto_id] -= cantidad
        
        venta = {
            'producto_id': producto_id,
            'producto': CATALOGO[producto_id],
            'cantidad': cantidad,
            'precio_unitario': precio_unitario,
            'descuento': descuento,
            'total': total,
            'fecha': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        ventas_buffer.append(venta)
        
        print(f"\n Venta registrada: {CATALOGO[producto_id]} x{cantidad} = \${total:.2f}")
    finally:
        # Este bloque siempre corre, haya salido bien o mal la venta
        print("Operación de registro de venta finalizada.\n")

<>:27: SyntaxWarning: invalid escape sequence '\$'
<>:55: SyntaxWarning: invalid escape sequence '\$'
<>:27: SyntaxWarning: invalid escape sequence '\$'
<>:55: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_563/2444673889.py:27: SyntaxWarning: invalid escape sequence '\$'
  print(f"Descuento del 5% aplicado: \${descuento:.2f}")
/tmp/ipykernel_563/2444673889.py:55: SyntaxWarning: invalid escape sequence '\$'
  print(f"\n Venta registrada: {CATALOGO[producto_id]} x{cantidad} = \${total:.2f}")


### 2.3 Guardar y cargar CSV (Pandas)

In [ ]:
def guardar_csv():
    """Guarda las ventas en un archivo CSV"""
    try:
        if not ventas_buffer:
            print("No hay ventas para guardar")
            return
        
        # Convertir a DataFrame
        df = pd.DataFrame(ventas_buffer)
        
        # Guardar CSV
        df.to_csv("ventas.csv", index=False, encoding='utf-8')
        print(f"Ventas guardadas en ventas.csv ({len(df)} registros)")
        
    except Exception as e:
        print(f"Error al guardar CSV: {e}")
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR CSV: {e}\n")

def cargar_csv():
    """Carga ventas desde un archivo CSV"""
    global ventas_buffer
    try:
        if not os.path.exists("ventas.csv"):
            print("Archivo ventas.csv no existe")
            return
        
        df = pd.read_csv("ventas.csv", encoding='utf-8')
        ventas_buffer = df.to_dict('records')
        print(f"Datos cargados: {len(ventas_buffer)} ventas")
        
    except FileNotFoundError:
        print("Archivo no encontrado")
    except Exception as e:
        print(f"Error al cargar CSV: {e}")
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR CARGA: {e}\n")

### 2.4 Analizar ventas (Pandas + NumPy)

`groupby` (Pandas) agrupa por producto para sumar ingresos y unidades. `np.sum`,
`np.mean` y `np.std` (NumPy) calculan las estadísticas generales. El descuento promedio
se calcula solo entre las ventas que sí tuvieron descuento, con la división protegida por
un `try/except ZeroDivisionError` para el caso en que ninguna venta haya calificado
todavía.

In [ ]:
def analizar_ventas():
    """Analiza las ventas usando Pandas y NumPy"""
    try:
        if not ventas_buffer:
            print("No hay ventas para analizar")
            return
        
    
        df = pd.DataFrame(ventas_buffer)
        
        totales = np.array(df['total'])
        
        print("\n" + "="*60)
        print("ANÁLISIS DE VENTAS")
        print("="*60)
        print(f"Total ventas: {len(df)}")
        print(f"Total ingresos: \${np.sum(totales):.2f}")
        print(f"Ingreso promedio: \${np.mean(totales):.2f}")
        print(f"Desviación estándar: \${np.std(totales):.2f}")
        
        print("\n INGRESOS POR PRODUCTO:")
        ingresos_por_producto = df.groupby('producto')['total'].sum().sort_values(ascending=False)
        for producto, ingreso in ingresos_por_producto.items():
            print(f" {producto}: \${ingreso:.2f}")
        
        # Producto más vendido
        producto_top = df.groupby('producto')['cantidad'].sum().idxmax()
        print(f"\n Producto más vendido: {producto_top}")
        
        # Descuento promedio solo entre las ventas que SI tuvieron descuento
        # (division por cero controlada: puede que ninguna venta haya calificado)
        ventas_con_descuento = df[df['descuento'] > 0]
        try:
            promedio_descuento = float(ventas_con_descuento['descuento'].sum()) / len(ventas_con_descuento)
        except ZeroDivisionError:
            print("\n No se aplicaron descuentos en ninguna venta todavia.")
        else:
            print(f"\n Descuento promedio en ventas con descuento (>=10 u.): \${promedio_descuento:.2f}")
        
        return df, ingresos_por_producto
        
    except Exception as e:
        print(f"Error en análisis: {e}")
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR ANALISIS: {e}\n")
        return None, None

<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:19: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
<>:38: SyntaxWarning: invalid escape sequence '\$'
<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:18: SyntaxWarning: invalid escape sequence '\$'
<>:19: SyntaxWarning: invalid escape sequence '\$'
<>:24: SyntaxWarning: invalid escape sequence '\$'
<>:38: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_563/1289543745.py:17: SyntaxWarning: invalid escape sequence '\$'
  print(f"Total ingresos: \${np.sum(totales):.2f}")
/tmp/ipykernel_563/1289543745.py:18: SyntaxWarning: invalid escape sequence '\$'
  print(f"Ingreso promedio: \${np.mean(totales):.2f}")
/tmp/ipykernel_563/1289543745.py:19: SyntaxWarning: invalid escape sequence '\$'
  print(f"Desviación estándar: \${np.std(totales):.2f}")
/tmp/ipykernel_563/1289543745.py:24: SyntaxWarning: invalid escape sequence '\$'
  

### 2.5 Gráfica de ingresos (Matplotlib) + Reto B

In [ ]:
AUTOR_GRAFICO = "Sheyla Tumbaco Morán"
MATERIA_GRAFICO = "Arquitectura de Computadoras y Sistemas Operativos"


def _construir_grafico_ingresos(ingresos_por_producto):
    """Crea la figura de ingresos por producto con estilo profesional
    (barras horizontales ordenadas, etiquetas en miles, pie de autor).
    Devuelve la figura y el eje ya listos para mostrar o guardar."""
    datos_ordenados = ingresos_por_producto.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 6), dpi=200)

    colores = plt.cm.Blues(np.linspace(0.45, 0.9, len(datos_ordenados)))
    barras = ax.barh(datos_ordenados.index, datos_ordenados.values,
                      color=colores, edgecolor='#1B3A4B', linewidth=0.8, height=0.62)

    for barra, valor in zip(barras, datos_ordenados.values):
        ax.text(valor + datos_ordenados.values.max() * 0.012,
                 barra.get_y() + barra.get_height() / 2,
                 f'\${valor:,.2f}', va='center', ha='left',
                 fontsize=10.5, color='#1B3A4B', fontweight='bold')

    ax.set_title('Ingresos por Producto - MiniTienda', fontsize=17,
                  fontweight='bold', color='#1B3A4B', pad=18)
    ax.set_xlabel('Ingresos (\$)', fontsize=12, color='#333333', labelpad=10)
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x:,.0f}'))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_color('#CCCCCC')
    ax.tick_params(axis='y', labelsize=11.5, colors='#1B3A4B', length=0)
    ax.tick_params(axis='x', labelsize=10, colors='#555555')
    ax.set_xlim(0, datos_ordenados.values.max() * 1.18)
    ax.grid(axis='x', color='#E0E0E0', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)

    fecha_hoy = datetime.now().strftime("%d/%m/%Y")
    fig.text(0.99, 0.01, f'{AUTOR_GRAFICO}  •  {MATERIA_GRAFICO}  •  {fecha_hoy}',
              ha='right', va='bottom', fontsize=8.5, color='#8A8A8A', style='italic')

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    return fig, ax


def graficar_ingresos():
    """Genera gráfica de ingresos por producto"""
    try:
        if not ventas_buffer:
            print("No hay datos para graficar")
            return

        df = pd.DataFrame(ventas_buffer)
        ingresos_por_producto = df.groupby('producto')['total'].sum()

        _construir_grafico_ingresos(ingresos_por_producto)
        plt.show()

    except Exception as e:
        print(f"Error al graficar: {e}")
        with open("log.txt", "a") as log_file:
            log_file.write(f"{datetime.now()} - ERROR GRAFICA: {e}\n")

def exportar_grafico_png():
    """Exporta el gráfico a PNG (RETO B)"""
    try:
        if not ventas_buffer:
            print("No hay datos para exportar")
            return

        df = pd.DataFrame(ventas_buffer)
        ingresos_por_producto = df.groupby('producto')['total'].sum()

        _construir_grafico_ingresos(ingresos_por_producto)
        plt.savefig("ingresos.png", dpi=200, bbox_inches='tight')
        print("Gráfico exportado como 'ingresos.png'")
        plt.close()

    except Exception as e:
        print(f"Error al exportar: {e}")

<>:20: SyntaxWarning: invalid escape sequence '\$'
<>:25: SyntaxWarning: invalid escape sequence '\$'
<>:27: SyntaxWarning: invalid escape sequence '\$'
<>:20: SyntaxWarning: invalid escape sequence '\$'
<>:25: SyntaxWarning: invalid escape sequence '\$'
<>:27: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_563/1292764051.py:20: SyntaxWarning: invalid escape sequence '\$'
  f'\${valor:,.2f}', va='center', ha='left',
/tmp/ipykernel_563/1292764051.py:25: SyntaxWarning: invalid escape sequence '\$'
  ax.set_xlabel('Ingresos (\$)', fontsize=12, color='#333333', labelpad=10)
/tmp/ipykernel_563/1292764051.py:27: SyntaxWarning: invalid escape sequence '\$'
  ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x:,.0f}'))


### 2.6 Agregar producto (Reto A)

In [ ]:
def agregar_producto():
    """Agrega un nuevo producto al catálogo (RETO A)"""
    try:
        print("\n" + "="*60)
        print("AGREGAR NUEVO PRODUCTO")
        print("="*60)
        
        nombre = input("Nombre del producto: ").strip()
        if not nombre:
            raise ValueError("El nombre no puede estar vacío")
        
        precio = float(input("Precio: \$"))
        if precio < 0:
            raise ValueError("El precio no puede ser negativo")
        
        stock = int(input("Stock inicial: "))
        if stock < 0:
            raise ValueError("El stock no puede ser negativo")
        
        # Nuevo ID
        nuevo_id = max(PRECIOS.keys()) + 1
        
        # Actualizar estructuras
        global CATALOGO
        CATALOGO = CATALOGO + (nombre,)
        PRECIOS[nuevo_id] = precio
        STOCK[nuevo_id] = stock
        
        print(f"Producto agregado: ID {nuevo_id} - {nombre}")
        
    except ValueError as e:
        print(f"Error: {e}")
    except Exception as e:
        print(f"Error inesperado: {e}")

<>:12: SyntaxWarning: invalid escape sequence '\$'
<>:12: SyntaxWarning: invalid escape sequence '\$'
/tmp/ipykernel_563/3072242374.py:12: SyntaxWarning: invalid escape sequence '\$'
  precio = float(input("Precio: \$"))


### 2.7 Generar datos de ejemplo (para pruebas rápidas)

In [ ]:
def generar_datos_ejemplo():
    """Genera datos de ejemplo para pruebas"""
    print("Generando datos de ejemplo...")
    productos_ejemplo = [
        (0, 2), (0, 1), (1, 5), (2, 3), 
        (3, 1), (4, 2), (1, 3), (2, 4),
        (0, 1), (3, 2)
    ]
    
    for prod_id, cant in productos_ejemplo:
        try:
            # Simular venta sin restar stock real
            venta = {
                'producto_id': prod_id,
                'producto': CATALOGO[prod_id],
                'cantidad': cant,
                'precio_unitario': PRECIOS[prod_id],
                'descuento': 0 if cant < 10 else PRECIOS[prod_id]*cant*0.05,
                'total': PRECIOS[prod_id]*cant * (0.95 if cant >= 10 else 1),
                'fecha': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
            ventas_buffer.append(venta)
        except:
            pass
    
    print(f"{len(ventas_buffer)} ventas generadas")

## 3. Menú principal

Controla todo con `while True`, dirige cada opción con `if/elif/else`, usa `continue`
cuando la entrada está vacía y `break` para salir con la opción 0. El `try/except` general
atrapa números inválidos, `Ctrl+C` y cualquier error inesperado sin cerrar el programa.

**Nota:** usa `input()`, así que está pensada para Colab/Jupyter interactivo. Para que el
notebook se pueda ejecutar de principio a fin de forma automática, la sección 4 simula el
mismo flujo llamando directamente a las funciones.

In [ ]:
def menu():
    """Menú principal del sistema"""
    while True:
        print("\n" + "="*60)
        print("MINITIENDA - SISTEMA DE VENTAS")
        print("="*60)
        print("1) Ver catálogo")
        print("2) Registrar venta")
        print("3) Guardar ventas (CSV)")
        print("4) Cargar ventas (CSV)")
        print("5) Analizar ventas")
        print("6) Mostrar gráfica")
        print("7) Exportar gráfica a PNG (RETO B)")
        print("8) Agregar producto (RETO A)")
        print("9) Generar datos de ejemplo")
        print("0) Salir")
        print("="*60)
        
        try:
            opcion = input("Seleccione una opción: ").strip()
            
            if not opcion:
                continue
                
            opcion = int(opcion)
            
            if opcion == 1:
                mostrar_catalogo()
                
            elif opcion == 2:
                registrar_venta()
                
            elif opcion == 3:
                guardar_csv()
                
            elif opcion == 4:
                cargar_csv()
                
            elif opcion == 5:
                analizar_ventas()
                
            elif opcion == 6:
                graficar_ingresos()
                
            elif opcion == 7:
                exportar_grafico_png()
                
            elif opcion == 8:
                agregar_producto()
                
            elif opcion == 9:
                generar_datos_ejemplo()
                
            elif opcion == 0:
                print("\n¡Gracias por usar MiniTienda!")
                break
                
            else:
                print("Opción no válida")
                
        except ValueError:
            print("Por favor, ingrese un número válido")
        except KeyboardInterrupt:
            print("\n¡Hasta luego!")
            break
        except Exception as e:
            print(f"Error: {e}")
            with open("log.txt", "a") as log_file:
                log_file.write(f"{datetime.now()} - ERROR MENU: {e}\n")

# Para jugar con el menu real en Colab/Jupyter, descomenta y ejecuta:
# menu()

## 4. Celdas de prueba (simulación sin `input()`)

Llaman directamente a las funciones para generar al menos 10 ventas y probar los dos
retos, sin escribir nada a mano. Sirven como evidencia automática de ejecución.

### 4.1 Catálogo inicial

In [ ]:
mostrar_catalogo()


 ID              PRODUCTO               PRECIO     STOCK  
  0   Laptop Dell XPS                \$ 1200.00       10
  1   Mouse Logitech                 \$   45.50       50
  2   Teclado Mecánico               \$   89.90       30
  3   Monitor Samsung                \$  350.00       15
  4   Audífonos Sony                 \$  120.00       25


### 4.2 Generar 10 ventas de ejemplo y guardarlas en CSV

In [ ]:
generar_datos_ejemplo()
guardar_csv()

Generando datos de ejemplo...
10 ventas generadas
Ventas guardadas en ventas.csv (10 registros)


### 4.3 Analizar ventas (Pandas + NumPy, incluye el caso sin descuentos)

In [ ]:
df_resultado, ingresos_resultado = analizar_ventas()


ANÁLISIS DE VENTAS
Total ventas: 10
Total ingresos: \$7083.30
Ingreso promedio: \$708.33
Desviación estándar: \$675.66

 INGRESOS POR PRODUCTO:
 Laptop Dell XPS: \$4800.00
 Monitor Samsung: \$1050.00
 Teclado Mecánico: \$629.30
 Mouse Logitech: \$364.00
 Audífonos Sony: \$240.00

 Producto más vendido: Mouse Logitech

 No se aplicaron descuentos en ninguna venta todavia.


### 4.4 Reto A - agregar un producto nuevo

In [ ]:
nombre_nuevo = "Bufanda"
precio_nuevo = 12.50
stock_nuevo = 40

nuevo_id = max(PRECIOS.keys()) + 1
CATALOGO = CATALOGO + (nombre_nuevo,)
PRECIOS[nuevo_id] = precio_nuevo
STOCK[nuevo_id] = stock_nuevo

print(f"Producto agregado: ID {nuevo_id} - {nombre_nuevo}")
mostrar_catalogo()

Producto agregado: ID 5 - Bufanda

 ID              PRODUCTO               PRECIO     STOCK  
  0   Laptop Dell XPS                \$ 1200.00       10
  1   Mouse Logitech                 \$   45.50       50
  2   Teclado Mecánico               \$   89.90       30
  3   Monitor Samsung                \$  350.00       15
  4   Audífonos Sony                 \$  120.00       25
  5   Bufanda                        \$   12.50       40


### 4.5 Reto B - exportar el gráfico a PNG

In [ ]:
exportar_grafico_png()

Gráfico exportado como 'ingresos.png'


### 4.6 Verificar `log.txt`

In [ ]:
if os.path.exists("log.txt"):
    with open("log.txt", "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Aun no se ha generado log.txt (no hubo errores registrados).")

Aun no se ha generado log.txt (no hubo errores registrados).


## 5. Preguntas de la asignación

**¿Qué parte la hizo Pandas? ¿Qué parte NumPy?**

Pandas: conversión del buffer de ventas a `DataFrame`, `groupby` para analizar las ventas
por producto, y lectura/escritura del CSV. NumPy: cálculo de estadísticas (`mean`, `std`,
`sum`) y operaciones vectorizadas sobre el arreglo de totales.

**¿Dónde usaste try/except y por qué?**

En `registrar_venta()` para manejar errores de entrada y validación de productos; en
`guardar_csv()` y `cargar_csv()` para manejar errores de archivos; en `analizar_ventas()`
para manejar errores de cálculo (incluyendo la división por cero controlada del descuento
promedio); en `graficar_ingresos()`/`exportar_grafico_png()` para errores al generar el
gráfico; y en `menu()` para manejar errores de entrada del usuario.

**¿Qué estructuras son tuplas, listas y diccionarios en el código?**

Tuplas: `CATALOGO` (productos inmutables). Listas: `ventas_buffer` (colección dinámica de
ventas). Diccionarios: `PRECIOS` y `STOCK` (accesos por ID).